In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

yolo_model = YOLO("yolo11n.pt")

VEHICLE_CLASSES = {"car", "truck", "bus", "motorcycle"}


def classify_vehicle(crop):
    return {
        "label": "test",
        "confidence": 0.0
    }


def run_anpr(frame):
    return {
        "plate": None
    }


def process_frame(frame):
    results = yolo_model(frame, conf=0.4, verbose=False)

    vehicles = []

    for result in results:
        if result.boxes is None:
            continue

        for box in result.boxes:
            confidence = float(box.conf[0])
            class_id = int(box.cls[0])
            class_name = result.names[class_id]

            if class_name.lower() not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = frame[y1:y2, x1:x2]

            vehicles.append({
                "bbox": (x1, y1, x2, y2),
                "class": class_name,
                "confidence": confidence,
                "classification": classify_vehicle(crop)
            })

    if vehicles:
        return {
            "route": "vehicle",
            "vehicles": vehicles
        }

    return {
        "route": "anpr",
        "result": run_anpr(frame)
    }


def test_frame(image_path):
    frame = cv2.imread(image_path)

    result = process_frame(frame)

    print("Route:", result["route"])

    if result["route"] == "vehicle":
        print("Vehicles:", len(result["vehicles"]))

        for i, vehicle in enumerate(result["vehicles"], 1):
            print(f"\nVehicle {i}")
            print("Class:", vehicle["class"])
            print("Confidence:", round(vehicle["confidence"], 2))
            print("Classification:", vehicle["classification"])

        display = frame.copy()

        for vehicle in result["vehicles"]:
            x1, y1, x2, y2 = vehicle["bbox"]

            cv2.rectangle(
                display,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            cv2.putText(
                display,
                f'{vehicle["class"]} {vehicle["confidence"]:.2f}',
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

        display = cv2.cvtColor(display, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 7))
        plt.imshow(display)
        plt.axis("off")
        plt.show()

    else:
        print("ANPR result:", result["result"])


test_frame("06sd6.jpg")

Route: anpr
ANPR result: {'plate': None}
